# Distillation sentiment v1 với hai phương án A và B

**A:** giữ độ phủ + chọn KNOWDIST theo tỷ số response loss của student.
**B:** giữ độ phủ, không cần chấm model, luôn chuẩn bị trước để dự phòng.
**R:** KNOWDIST random làm đối chứng. ICLDIST và dev giống nhau giữa cả ba.

Đây là pipeline cần kiểm chứng GPU, không có bảo đảm tăng F1 hoặc tính mới.
Mặc định chỉ chuẩn bị B/R và chạy dry-run; chưa train cho đến khi RUN_TRAIN=True.
Chuyển ACTIVE_PLAN mà giữ thư mục riêng; không nối K của A sang I của B.
Lần đầu chạy theo cell và đọc preview. Sau khi có checkpoint, chạy lại bootstrap,
config, runtime và cell train; không chạy lại các cell tạo output không rỗng.
Lưu full outputs bằng Save Version. ZIP adapter không đủ resume.


In [ ]:
from pathlib import Path
import json, os, sys, subprocess, importlib.metadata
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
BASE = Path('/kaggle/working/sentiment-v1-ab')
PROJECT = BASE / 'source'
PROJECT.mkdir(parents=True, exist_ok=True)

SOURCES = {'training/distill_core.py': '"""Dependency-free data checks, splits and adaptive group weights."""\nimport hashlib\nimport json\nimport math\nimport random\nfrom collections import Counter, defaultdict\nfrom pathlib import Path\n\n\ndef digest(value):\n    return hashlib.sha256(json.dumps(value, ensure_ascii=False, sort_keys=True).encode()).hexdigest()\n\n\ndef file_hash(path):\n    h = hashlib.sha256()\n    with open(path, \'rb\') as stream:\n        for chunk in iter(lambda: stream.read(1024 * 1024), b\'\'):\n            h.update(chunk)\n    return h.hexdigest()\n\n\ndef read_records(path):\n    with open(path, encoding=\'utf-8-sig\') as stream:\n        for line, text in enumerate(stream, 1):\n            if not text.strip():\n                continue\n            try:\n                obj = json.loads(text)\n                if not isinstance(obj, dict):\n                    raise ValueError(\'record is not an object\')\n                yield line, obj\n            except (ValueError, TypeError) as exc:\n                raise ValueError(f\'Invalid JSON record at line {line}: {exc}\') from exc\n\n\ndef mapped_record(obj, prompt_fields, response_field, group_field, source_field):\n    """Explicit field order; preserve each field\'s contents, including whitespace."""\n    parts = []\n    for field in prompt_fields:\n        value = obj.get(field)\n        if value is None:\n            raise ValueError(f\'missing prompt field: {field}\')\n        if not isinstance(value, str):\n            raise ValueError(f\'prompt field must be a string: {field}\')\n        if value.strip():\n            parts.append(value)\n    response = obj.get(response_field)\n    if not parts or not isinstance(response, str) or not response.strip():\n        raise ValueError(\'empty prompt or response\')\n    group = obj.get(group_field)\n    if group is None or not str(group).strip():\n        raise ValueError(f\'missing group field: {group_field}\')\n    source = obj.get(source_field)\n    if not isinstance(source, str) or not source.strip():\n        raise ValueError(f\'missing source/input field for group split: {source_field}\')\n    # Normalize only the split key; never alter the training target.\n    source_key = \' \'.join(source.split()).casefold()\n    return {\'prompt\': \'\\n\\n\'.join(parts), \'response\': response,\n            \'group\': str(group), \'source_key\': digest(source_key)}\n\n\ndef split_records(records, seed, dev_fraction=0.1, max_train=0, probe_per_group=4,\n                  holdout_per_group=4, min_group_train=8, max_groups=32):\n    if not 0 < dev_fraction < 1:\n        raise ValueError(\'dev_fraction must be between 0 and 1\')\n    if min(probe_per_group, holdout_per_group, min_group_train, max_groups) < 1 or max_train < 0:\n        raise ValueError(\'Invalid split limits\')\n    # Assign every input group before selecting a pilot subset. No input crosses splits.\n    keys = sorted({r[\'source_key\'] for r in records})\n    random.Random(seed).shuffle(keys)\n    ndev = max(2, round(len(keys) * dev_fraction))\n    if ndev >= len(keys):\n        raise ValueError(\'Not enough independent inputs to split\')\n    dev_keys = set(keys[:ndev])\n    probe_keys = set(keys[:ndev // 2])\n    train = [r for r in records if r[\'source_key\'] not in dev_keys]\n    random.Random(seed).shuffle(train)\n    if max_train:\n        train = train[:max_train]\n    counts = Counter(r[\'group\'] for r in train)\n    # Group collapsing uses TRAIN only. Reserve one slot for rare groups.\n    common = [g for g, n in sorted(counts.items(), key=lambda x: (-x[1], x[0]))\n              if n >= min_group_train][:max_groups - 1]\n    mapping = {g: g if g in common else \'__rare__\' for g in counts}\n    groups = sorted(set(mapping.values()))\n    for r in train:\n        r[\'train_group\'] = mapping[r[\'group\']]\n    probe, holdout = [], []\n    for allowed, dest, limit in ((probe_keys, probe, probe_per_group),\n                                  (dev_keys - probe_keys, holdout, holdout_per_group)):\n        buckets = defaultdict(list)\n        for r in records:\n            if r[\'source_key\'] in allowed and r[\'group\'] in mapping:\n                r[\'train_group\'] = mapping[r[\'group\']]\n                buckets[r[\'train_group\']].append(r)\n        for g in groups:\n            items = sorted(buckets[g], key=lambda r: r[\'id\'])\n            random.Random(f\'{seed}:{g}\').shuffle(items)\n            dest.extend(items[:limit])\n    if not train or not probe or not holdout:\n        raise ValueError(\'Empty train/probe/holdout; increase pilot size or dev fraction\')\n    return train, probe, holdout, mapping\n\n\nclass GroupWeights:\n    """Progress normalized by initial per-response CE; bounded mean-one weights."""\n    def __init__(self, groups, counts, mode=\'progress\', mix=0.5, temperature=0.05,\n                 ema=0.5, floor=0.5, ceiling=2.0):\n        if mode not in (\'uniform\', \'balanced\', \'loss\', \'progress\'):\n            raise ValueError(\'Unknown weighting mode\')\n        if not 0 <= mix <= 1 or temperature <= 0 or not 0 <= ema < 1:\n            raise ValueError(\'Invalid mix/temperature/EMA\')\n        if not 0 < floor <= 1 <= ceiling:\n            raise ValueError(\'Require 0 < floor <= 1 <= ceiling\')\n        self.groups = list(groups)\n        self.counts = dict(counts)\n        self.mode, self.mix, self.temperature, self.ema = mode, mix, temperature, ema\n        self.floor, self.ceiling = floor, ceiling\n        self.initial, self.previous, self.smoothed = {}, {}, {}\n        self.weights = {g: 1.0 for g in groups}\n        if mode == \'balanced\':\n            self.weights = self._bounded({g: 1 / counts[g] for g in groups})\n\n    def _bounded(self, raw):\n        # Solve E_train[clip(scale * raw, floor, ceiling)] = 1.\n        total = sum(self.counts.values())\n        lo, hi = 0., 1.\n        def values(scale):\n            return {g: min(self.ceiling, max(self.floor, scale * raw[g])) for g in self.groups}\n        def mean(w):\n            return sum(w[g] * self.counts[g] for g in self.groups) / total\n        while mean(values(hi)) < 1 - 1e-12:\n            hi *= 2\n        for _ in range(80):\n            mid = (lo + hi) / 2\n            if mean(values(mid)) < 1:\n                lo = mid\n            else:\n                hi = mid\n        return values((lo + hi) / 2)\n\n    def update(self, losses):\n        if not losses or any(not math.isfinite(v) or v < 0 for v in losses.values()):\n            raise ValueError(\'Probe losses must be nonnegative and finite\')\n        scores = {}\n        for g, value in losses.items():\n            if g not in self.weights:\n                raise ValueError(f\'Unknown probe group: {g}\')\n            if g not in self.initial:\n                self.initial[g] = self.previous[g] = self.smoothed[g] = value\n                scores[g] = 0.\n                continue\n            smooth = self.ema * self.smoothed[g] + (1 - self.ema) * value\n            progress = max(0., (self.smoothed[g] - smooth) / max(self.initial[g], 1e-8))\n            scores[g] = progress if self.mode == \'progress\' else smooth / max(self.initial[g], 1e-8)\n            self.previous[g], self.smoothed[g] = self.smoothed[g], smooth\n        if self.mode in (\'uniform\', \'balanced\'):\n            return self.weights\n        # Groups without probes retain a neutral score; coverage is reported.\n        scores = {g: scores.get(g, 0.) for g in self.groups}\n        peak = max(scores.values())\n        exp = {g: math.exp((v - peak) / self.temperature) for g, v in scores.items()}\n        avg = sum(exp.values()) / len(exp)\n        raw = {g: (1 - self.mix) + self.mix * exp[g] / avg for g in self.groups}\n        self.weights = self._bounded(raw)\n        return self.weights\n\n\ndef write_json(path, value):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temp = path.with_suffix(path.suffix + \'.tmp\')\n    temp.write_text(json.dumps(value, ensure_ascii=False, indent=2), encoding=\'utf-8\')\n    temp.replace(path)\n', 'training/v1_core.py': '"""Dependency-free v1 normalization and deterministic budgeted selection.\n\nScores are ranking proxies, never certificates of teacher correctness.\n"""\nfrom collections import Counter, defaultdict\nimport heapq\nimport json\nimport math\nfrom pathlib import Path\nimport random\nimport re\n\nfrom distill_core import digest\n\n\ndef iter_objects(path):\n    """Stream a JSON array or JSONL. Malformed files stop the run."""\n    path = Path(path)\n    with path.open(encoding=\'utf-8-sig\') as stream:\n        first = \'\'\n        while not first:\n            char = stream.read(1)\n            if not char:\n                return\n            if not char.isspace():\n                first = char\n        if first != \'[\':\n            stream.seek(0)\n            for number, line in enumerate(stream, 1):\n                if line.strip():\n                    obj = json.loads(line)\n                    if not isinstance(obj, dict):\n                        raise ValueError(f\'{path}:{number}: expected object\')\n                    yield number, obj\n            return\n        decoder, buffer, number, expecting_value = json.JSONDecoder(), \'\', 0, True\n        eof = False\n        while True:\n            buffer = buffer.lstrip()\n            if not buffer and not eof:\n                chunk = stream.read(65536)\n                eof = not chunk\n                buffer += chunk\n                continue\n            if buffer.startswith(\']\'):\n                if expecting_value and number:\n                    raise ValueError(f\'{path}: trailing comma\')\n                if (buffer[1:] + stream.read()).strip():\n                    raise ValueError(f\'{path}: trailing content\')\n                return\n            if not expecting_value:\n                if not buffer.startswith(\',\'):\n                    raise ValueError(f\'{path}: missing comma or closing bracket\')\n                buffer, expecting_value = buffer[1:], True\n                continue\n            try:\n                obj, end = decoder.raw_decode(buffer)\n            except ValueError:\n                if eof:\n                    raise ValueError(f\'{path}: invalid or incomplete JSON array\') from None\n                chunk = stream.read(65536)\n                eof = not chunk\n                buffer += chunk\n                continue\n            if not isinstance(obj, dict):\n                raise ValueError(f\'{path}: expected object at index {number}\')\n            number += 1\n            yield number, obj\n            buffer, expecting_value = buffer[end:], False\n\n\ndef normalized_text(text):\n    return \' \'.join(text.split()).casefold()\n\n\ndef source_text(prompt, stage):\n    """Conservative, logged extraction. Unknown templates are excluded, not guessed.\n\n    KNOW: final Review/Text/Tweet section. ICL: last supported input marker,\n    retaining auxiliary fields (e.g. Edit) and stripping only terminal Output.\n    This isolates the final query, NOT texts inside demonstrations.\n    """\n    labels = (\'Review|Text|Tweet\' if stage == \'knowdist\'\n              else \'Sentence|Sentence1|Premise|Input|Text|Review|Tweet|News Headline\')\n    matches = list(re.finditer(rf\'(?m)^(?:{labels}):[ \\t]*\', prompt))\n    if not matches:\n        raise ValueError(\'source_unresolved: unsupported input marker; inspect mapping\')\n    value = prompt[matches[-1].end():]\n    if stage == \'icldist\':\n        value = re.sub(r\'\\n(?:Output|Label|Answer):[ \\t]*\\Z\', \'\', value)\n    if not value.strip():\n        raise ValueError(\'source_unresolved: empty final input\')\n    return value\n\n\ndef canonical_record(obj, relative_file, index, source_sha):\n    if \'prompt\' in obj and \'llm_response\' in obj:\n        stage, prompt, response = \'knowdist\', obj[\'prompt\'], obj[\'llm_response\']\n        kind = obj.get(\'instr_type\') or Path(relative_file).stem.removeprefix(\'sample_\')\n        if not str(kind).startswith((\'analysis_\', \'rewrite_\')):\n            raise ValueError(\'unknown KNOWDIST prompt type; supply instr_type\')\n    elif \'instruction\' in obj and \'output\' in obj:\n        stage, response = \'icldist\', obj[\'output\']\n        instruction, extra = obj[\'instruction\'], obj.get(\'input\', \'\')\n        if not isinstance(instruction, str) or not isinstance(extra, str):\n            raise ValueError(\'instruction/input must be strings\')\n        prompt = instruction + (\'\\n\\n\' + extra if extra.strip() else \'\')\n        kind = (\'task_diversification\' if \'task_diversification\' in relative_file\n                else \'sentiment_alignment\' if \'sentiment_distillation\' in relative_file else None)\n        if kind is None:\n            raise ValueError(\'unknown ICLDIST file; cannot identify TD vs sentiment\')\n    else:\n        raise ValueError(\'unknown_schema\')\n    if not isinstance(prompt, str) or not prompt.strip() or not isinstance(response, str) or not response.strip():\n        raise ValueError(\'empty_or_nonstring_prompt_response\')\n    source = obj.get(\'source_text\')\n    method = \'explicit_source_text\' if isinstance(source, str) and source.strip() else \'final_input_marker_v1\'\n    if method != \'explicit_source_text\' and stage == \'icldist\' and obj.get(\'input\', \'\').strip():\n        source, method = obj[\'input\'], \'explicit_input_field\'\n    if method == \'final_input_marker_v1\':\n        source = source_text(prompt, stage)\n    name = str(obj.get(\'dataset_name\', \'unknown\'))\n    return {\'id\': f\'{source_sha}:{index}\', \'stage\': stage, \'prompt\': prompt,\n            \'response\': response, \'source_key\': digest(normalized_text(source)),\n            \'source_text\': source, \'source_extraction\': method,\n            \'group\': f\'{kind}:{name}\', \'kind\': kind, \'source_dataset\': name,\n            \'file\': relative_file, \'position\': index}\n\n\ndef split_name(source_key, seed, dev_fraction):\n    if not 0 < dev_fraction < 1:\n        raise ValueError(\'dev_fraction must be in (0,1)\')\n    value = int(digest([seed, source_key])[:16], 16) / 2**64\n    return \'probe\' if value < dev_fraction / 2 else \'holdout\' if value < dev_fraction else \'train\'\n\n\ndef limited_candidates(rows, seed, per_group, caps):\n    """Stable hash reservoirs; bounded memory per (stage,split,group).\n\n    Train gets a total per-stage cap using round-robin group coverage. Dev has\n    its own per-group cap, independent of the selection method.\n    """\n    if per_group < 1 or any(v < 1 for v in caps.values()):\n        raise ValueError(\'candidate limits must be positive\')\n    buckets = defaultdict(list)\n    for r in rows:\n        key = (r[\'stage\'], r[\'split\'], r[\'group\'])\n        limit = per_group if r[\'split\'] == \'train\' else 8\n        priority = int(digest([seed, r[\'id\']]), 16)\n        entry = (-priority, r[\'id\'], r)\n        heap = buckets[key]\n        if len(heap) < limit:\n            heapq.heappush(heap, entry)\n        elif priority < -heap[0][0]:\n            heapq.heapreplace(heap, entry)\n    result = []\n    for stage in (\'knowdist\', \'icldist\'):\n        train = []\n        for (s, split, _), entries in sorted(buckets.items()):\n            if s != stage:\n                continue\n            ordered = [v[2] for v in sorted(entries, key=lambda x: (-x[0], x[1]))]\n            if split == \'train\':\n                train.append(ordered)\n            else:\n                result.extend(ordered)\n        count = 0\n        for index in range(max(map(len, train), default=0)):\n            for group in train:\n                if index < len(group) and count < caps[stage]:\n                    result.append(group[index])\n                    count += 1\n    return result\n\n\ndef select_rows(rows, method, token_budget, max_records, seed, scores=None, min_ifd=0., max_ifd=1.):\n    """B is coverage-balanced; A ranks IFD within each group. No test feedback.\n\n    Bounds are explicit experimental controls, NOT verified quality thresholds.\n    Shared eligibility/candidate pool, strict no-overshoot selection budget.\n    """\n    if method not in (\'random\', \'coverage\', \'student\') or token_budget < 1 or max_records < 1:\n        raise ValueError(\'invalid selection controls\')\n    if not 0 <= min_ifd < max_ifd or not math.isfinite(max_ifd):\n        raise ValueError(\'invalid IFD interval\')\n    if len({r[\'id\'] for r in rows}) != len(rows):\n        raise ValueError(\'duplicate candidate IDs\')\n    if any(r[\'split\'] != \'train\' for r in rows):\n        raise ValueError(\'selection must use train only\')\n    buckets = defaultdict(list)\n    for r in sorted(rows, key=lambda r: r[\'id\']):\n        if len(r[\'input_ids\']) < 2 or not any(t != -100 for t in r[\'labels\'][1:]):\n            raise ValueError(\'invalid tokenized record\')\n        if method == \'student\':\n            if scores is None or r[\'id\'] not in scores:\n                raise ValueError(f\'missing score for {r["id"]}; no silent fallback\')\n            score = scores[r[\'id\']]\n            if not math.isfinite(score):\n                raise ValueError(\'non-finite IFD\')\n            if not min_ifd <= score <= max_ifd:\n                continue\n        buckets[r[\'group\'] if method != \'random\' else \'__all__\'].append(r)\n    for group, values in buckets.items():\n        random.Random(f\'{seed}:{group}\').shuffle(values)\n        if method == \'student\':\n            values.sort(key=lambda r: -scores[r[\'id\']])\n    selected, tokens, group_tokens = [], 0, Counter()\n    cursors = Counter()\n    while buckets and len(selected) < max_records:\n        # Token coverage, rather than equal record count, avoids long groups dominating.\n        group = min(buckets, key=lambda g: (group_tokens[g], g))\n        values = buckets[group]\n        chosen = None\n        while cursors[group] < len(values):\n            r = values[cursors[group]]\n            cursors[group] += 1\n            if tokens + len(r[\'input_ids\']) <= token_budget:\n                chosen = r\n                break\n        if chosen is None:\n            del buckets[group]\n            continue\n        selected.append(chosen)\n        size = len(chosen[\'input_ids\'])\n        tokens += size\n        group_tokens[group] += size\n    if not selected:\n        raise ValueError(\'empty selection; inspect score interval/budget/candidate pool\')\n    return selected, {\'method\': method, \'tokens\': tokens, \'token_budget\': token_budget,\n                      \'unused_tokens\': token_budget - tokens, \'records\': len(selected),\n                      \'groups\': dict(Counter(r[\'group\'] for r in selected)),\n                      \'group_tokens\': dict(group_tokens),\n                      \'missing_groups\': sorted({r[\'group\'] for r in rows} - {r[\'group\'] for r in selected})}\n\n\ndef steps_for_tokens(rows, target, batch_size, accumulation, seed):\n    """Simulate trainer\'s exact shuffled stream up to an optimizer boundary."""\n    if not rows or min(target, batch_size, accumulation) < 1:\n        raise ValueError(\'invalid training budget\')\n    epoch, offset, steps, tokens, order = 0, 0, 0, 0, []\n    sizes = [len(r[\'input_ids\']) for r in rows]\n    while tokens < target:\n        for _ in range(batch_size * accumulation):\n            if not order:\n                order = list(range(len(rows)))\n                random.Random(seed + epoch).shuffle(order)\n            tokens += sizes[order[offset]]\n            offset += 1\n            if offset == len(order):\n                epoch, offset, order = epoch + 1, 0, []\n        steps += 1\n    return steps, tokens\n', 'training/train_distill.py': '"""Kaggle pilot: offline-response distillation, with uniform/adaptive group loss.\n\nNo teacher model, generated labels, or benchmark test feedback is used.\nHeavy dependencies are imported only by prepare/train/export commands.\n"""\nimport os\nos.environ.setdefault(\'PYTORCH_CUDA_ALLOC_CONF\', \'expandable_segments:True\')\nimport argparse\nfrom collections import Counter, defaultdict\nfrom importlib.metadata import version\nimport json\nfrom pathlib import Path\nimport random\nimport sys\nimport time\n\nfrom distill_core import (GroupWeights, digest, file_hash, mapped_record, read_records,\n                          split_records, write_json)\n\n\ndef audit(args):\n    fields, groups, examples = Counter(), Counter(), []\n    count = 0\n    rng = random.Random(42)\n    for line, obj in read_records(args.file):\n        count += 1\n        fields.update(obj.keys())\n        groups[str(obj.get(args.group_field, \'<missing>\'))] += 1\n        if len(examples) < args.examples:\n            examples.append({\'line\': line, \'record\': obj})\n        else:\n            index = rng.randrange(count)\n            if index < args.examples:\n                examples[index] = {\'line\': line, \'record\': obj}\n    report = {\'file\': str(Path(args.file).resolve()), \'sha256\': file_hash(args.file),\n              \'records\': count, \'fields\': dict(fields), \'groups\': dict(groups), \'examples\': examples}\n    write_json(args.output, report)\n    print(json.dumps(report, ensure_ascii=False, indent=2))\n\n\ndef prepare(args):\n    if not args.mapping_confirmed:\n        raise ValueError(\'Inspect audit.json first; then explicitly pass --mapping_confirmed with verified field names.\')\n    if args.max_length < 2:\n        raise ValueError(\'max_length must include prompt and response\')\n    out = Path(args.output)\n    if out.exists() and any(out.iterdir()):\n        raise ValueError(\'Prepared output must be empty; choose a new directory to preserve prior manifests.\')\n    from huggingface_hub import HfApi\n    from transformers import AutoTokenizer\n    revision = HfApi().model_info(args.model, revision=args.model_revision).sha\n    tokenizer = AutoTokenizer.from_pretrained(args.model, revision=revision)\n    if tokenizer.eos_token_id is None or not tokenizer.chat_template:\n        raise ValueError(\'Tokenizer requires chat template and EOS\')\n    file_path = Path(args.file)\n    if not file_path.exists():\n        from huggingface_hub import hf_hub_download\n        print(f"Corpus file not found at {args.file}, downloading from Hugging Face hub...")\n        fname = file_path.name if file_path.name.endswith(\'.jsonl\') else "Qwen3-32B_50k.jsonl"\n        downloaded = hf_hub_download(\n            repo_id="zhang-yice/sentiment-distillation-v2",\n            filename=fname,\n            repo_type="dataset",\n            revision=args.corpus_revision\n        )\n        file_path = Path(downloaded)\n        print(f"Corpus downloaded and resolved to: {file_path}")\n    records, rejected, seen = [], [], set()\n    source_sha = file_hash(file_path)\n    fields = args.prompt_fields.split(\',\')\n    for line, obj in read_records(file_path):\n        record_id = f\'{source_sha}:{line}\'\n        try:\n            r = mapped_record(obj, fields, args.response_field, args.group_field, args.source_field)\n            key = digest([r[\'prompt\'], r[\'response\'], r[\'group\']])\n            if key in seen:\n                raise ValueError(\'duplicate prompt/response/group\')\n            seen.add(key)\n            prefix = tokenizer.apply_chat_template(\n                [{\'role\': \'user\', \'content\': r[\'prompt\']}], tokenize=False,\n                add_generation_prompt=True, enable_thinking=False)\n            prompt_ids = tokenizer.encode(prefix, add_special_tokens=False)\n            response_ids = tokenizer.encode(r[\'response\'], add_special_tokens=False)\n            if not response_ids:\n                raise ValueError(\'empty response tokens\')\n            if response_ids[-1] != tokenizer.eos_token_id:\n                response_ids.append(tokenizer.eos_token_id)\n            if len(prompt_ids) + len(response_ids) > args.max_length:\n                raise ValueError(\'overlength: whole record excluded; no response truncation\')\n            r.update(id=record_id, input_ids=prompt_ids + response_ids,\n                     labels=[-100] * len(prompt_ids) + response_ids,\n                     prompt_tokens=len(prompt_ids), response_tokens=len(response_ids))\n            records.append(r)\n        except ValueError as exc:\n            rejected.append({\'id\': record_id, \'reason\': str(exc)})\n    train, probe, holdout, mapping = split_records(\n        records, args.seed, args.dev_fraction, args.max_train, args.probe_per_group,\n        args.holdout_per_group, args.min_group_train, args.max_groups)\n    out.mkdir(parents=True, exist_ok=True)\n    for name, rows in [(\'train\', train), (\'probe\', probe), (\'holdout\', holdout)]:\n        with (out / f\'{name}.jsonl\').open(\'w\', encoding=\'utf-8\') as stream:\n            for r in rows:\n                stream.write(json.dumps(r, ensure_ascii=False) + \'\\n\')\n    tokenizer.save_pretrained(out / \'tokenizer\')\n    clean_config = {k: v for k, v in vars(args).items() if k != \'func\'}\n    clean_config[\'file\'] = str(file_path)\n    manifest = {\'config\': clean_config, \'model\': args.model, \'model_revision\': revision,\n                \'source_sha256\': source_sha, \'corpus_revision\': args.corpus_revision,\n                \'tokenizer_template_sha256\': digest(tokenizer.chat_template), \'group_mapping\': mapping,\n                \'tokenizer_hashes\': {p.name: file_hash(p) for p in sorted((out / \'tokenizer\').iterdir()) if p.is_file()},\n                \'eligible\': len(records), \'rejected\': len(rejected),\n                \'counts\': {name: len(rows) for name, rows in [(\'train\', train), (\'probe\', probe), (\'holdout\', holdout)]},\n                \'train_groups\': dict(Counter(r[\'train_group\'] for r in train)),\n                \'probe_groups\': dict(Counter(r[\'train_group\'] for r in probe)),\n                \'holdout_groups\': dict(Counter(r[\'train_group\'] for r in holdout)),\n                \'split_hashes\': {name: file_hash(out / f\'{name}.jsonl\') for name in (\'train\', \'probe\', \'holdout\')},\n                \'train_input_tokens\': sum(len(r[\'input_ids\']) for r in train),\n                \'train_response_tokens\': sum(r[\'response_tokens\'] for r in train)}\n    write_json(out / \'rejected.json\', rejected)\n    write_json(out / \'manifest.json\', manifest)\n    write_json(out / \'preview.json\', [{k: r[k] for k in (\'id\', \'prompt\', \'response\', \'group\', \'train_group\')}\n                                     for r in train[:12]])\n    print(json.dumps(manifest, indent=2, ensure_ascii=False))\n    print(\'Inspect preview.json, exclusions and group coverage before training.\')\n\n\ndef per_example_loss(logits, labels):\n    """Response-only mean CE, chunked to avoid fp32 [batch, sequence, vocab]."""\n    import torch\n    import torch.nn.functional as F\n    result = []\n    for row, target in zip(logits, labels):\n        target = target[1:]\n        positions = torch.where(target != -100)[0]\n        if not positions.numel():\n            raise ValueError(\'Sample has no response targets\')\n        total = row.new_zeros((), dtype=torch.float32)\n        for indices in positions.split(128):\n            total = total + F.cross_entropy(row[indices].float(), target[indices], reduction=\'sum\')\n        result.append(total / positions.numel())\n    return torch.stack(result)\n\n\ndef _disable_torchao():\n    """Kaggle pre-installs torchao==0.10.0 which triggers PEFT >= 0.16.0 check crash."""\n    import sys\n    try:\n        import peft.import_utils\n        peft.import_utils.is_torchao_available = lambda: False\n    except Exception:\n        pass\n    try:\n        import peft.tuners.lora.torchao\n        peft.tuners.lora.torchao.is_torchao_available = lambda: False\n        peft.tuners.lora.torchao.dispatch_torchao = lambda *args, **kwargs: None\n    except Exception:\n        pass\n    try:\n        import peft.tuners.lora.layer\n        peft.tuners.lora.layer.is_torchao_available = lambda: False\n    except Exception:\n        pass\n    try:\n        import peft.tuners.lora.model\n        peft.tuners.lora.model.is_torchao_available = lambda: False\n    except Exception:\n        pass\n    for mod in list(sys.modules.values()):\n        if hasattr(mod, \'is_torchao_available\'):\n            try:\n                mod.is_torchao_available = lambda: False\n            except Exception:\n                pass\n\n\ndef train(args):\n    import os\n    os.environ.setdefault(\'CUBLAS_WORKSPACE_CONFIG\', \':4096:8\')\n    import numpy as np\n    import torch\n    _disable_torchao()\n    from peft import LoraConfig, PeftModel, get_peft_model, prepare_model_for_kbit_training\n    _disable_torchao()\n    from transformers import (AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig,\n                              get_cosine_schedule_with_warmup, set_seed)\n    if not torch.cuda.is_available():\n        raise ValueError(\'Train on Kaggle/Colab CUDA GPU; CPU training disabled\')\n    if int(os.environ.get(\'WORLD_SIZE\', \'1\')) != 1:\n        raise ValueError(\'This pilot supports one process/GPU; do not launch with torchrun\')\n    if min(args.steps, args.batch_size, args.grad_accum, args.probe_every, args.save_every, args.log_every, args.rank) < 1:\n        raise ValueError(\'steps, batch, accumulation and intervals must be positive\')\n    if args.learning_rate <= 0 or not 0 <= args.warmup_ratio <= 1 or args.max_seconds < 0:\n        raise ValueError(\'Invalid learning rate/warmup\')\n    # When resuming stage two, init_adapter is retained as hashed ancestry in\n    # identity; checkpoint weights below take precedence over initialization.\n    prepared, out = Path(args.prepared), Path(args.output)\n    manifest = json.loads((prepared / \'manifest.json\').read_text())\n    for split, expected in manifest[\'split_hashes\'].items():\n        if file_hash(prepared / f\'{split}.jsonl\') != expected:\n            raise ValueError(f\'Prepared split changed: {split}\')\n    for name, expected in manifest[\'tokenizer_hashes\'].items():\n        if file_hash(prepared / \'tokenizer\' / name) != expected:\n            raise ValueError(f\'Prepared tokenizer changed: {name}\')\n    data = {name: [r for _, r in read_records(prepared / f\'{name}.jsonl\')]\n            for name in (\'train\', \'probe\', \'holdout\')}\n    tokenizer = AutoTokenizer.from_pretrained(prepared / \'tokenizer\')\n    tokenizer.pad_token = tokenizer.eos_token\n    set_seed(args.seed)\n    torch.backends.cuda.matmul.allow_tf32 = False\n    torch.backends.cudnn.benchmark = False\n    bf16 = torch.cuda.get_device_capability(0)[0] >= 8 and torch.cuda.is_bf16_supported()\n    dtype = torch.bfloat16 if bf16 else torch.float16\n    versions = {p: version(p) for p in (\'torch\', \'transformers\', \'peft\', \'accelerate\', \'numpy\')}\n    if args.four_bit:\n        versions[\'bitsandbytes\'] = version(\'bitsandbytes\')\n    config = {k: v for k, v in vars(args).items() if k not in (\'output\', \'resume\', \'func\', \'max_seconds\')}\n    if args.init_adapter:\n        adapter_path = Path(args.init_adapter)\n        parent_config = json.loads((adapter_path / \'adapter_config.json\').read_text())\n        if parent_config.get(\'base_model_name_or_path\') != manifest[\'model\']:\n            raise ValueError(\'Initial adapter base model differs from prepared model\')\n        if parent_config.get(\'r\') != args.rank or parent_config.get(\'lora_alpha\') != args.rank * 2:\n            raise ValueError(\'Initial adapter rank/alpha differs\')\n        parent_files = {p.name: file_hash(p) for p in sorted(adapter_path.iterdir()) if p.is_file()}\n        if not any(name.endswith((\'.safetensors\', \'.bin\')) for name in parent_files):\n            raise ValueError(\'Initial adapter has no weight file\')\n        config[\'init_adapter_hashes\'] = parent_files\n    identity = {\'config\': config, \'manifest_sha256\': file_hash(prepared / \'manifest.json\'),\n                \'source_sha256\': digest([file_hash(Path(__file__)), file_hash(Path(__file__).with_name(\'distill_core.py\'))]),\n                \'versions\': versions, \'python\': sys.version, \'gpu\': torch.cuda.get_device_name(0), \'dtype\': str(dtype)}\n    if out.exists() and any(out.iterdir()) and not args.resume:\n        raise ValueError(\'Output is not empty. Use --resume or choose a new run directory.\')\n    checkpoint = None\n    if args.resume:\n        checkpoint = Path(args.resume).resolve()\n        if checkpoint.parent != out.resolve():\n            raise ValueError(\'Resume in the original run output directory\')\n        if json.loads((checkpoint / \'identity.json\').read_text()) != identity:\n            raise ValueError(\'Resume configuration/data/source/environment differs\')\n    out.mkdir(parents=True, exist_ok=True)\n    write_json(out / \'identity.json\', identity)\n    options = {\'revision\': manifest[\'model_revision\'], \'torch_dtype\': dtype,\n               \'device_map\': {\'\': 0}, \'attn_implementation\': \'sdpa\'}\n    if args.four_bit:\n        options[\'quantization_config\'] = BitsAndBytesConfig(\n            load_in_4bit=True, bnb_4bit_quant_type=\'nf4\', bnb_4bit_use_double_quant=True,\n            bnb_4bit_compute_dtype=dtype)\n    model = AutoModelForCausalLM.from_pretrained(manifest[\'model\'], **options)\n    model.config.use_cache = False\n    if args.four_bit:\n        model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)\n    else:\n        model.gradient_checkpointing_enable()\n        model.enable_input_require_grads()\n    if checkpoint or args.init_adapter:\n        model = PeftModel.from_pretrained(model, checkpoint / \'adapter\' if checkpoint else args.init_adapter,\n                                         is_trainable=True)\n    else:\n        target_mods = [m.strip() for m in args.target_modules.split(\',\') if m.strip()]\n        model = get_peft_model(model, LoraConfig(\n            task_type=\'CAUSAL_LM\', r=args.rank, lora_alpha=args.rank * 2,\n            lora_dropout=0., target_modules=target_mods))\n    model.print_trainable_parameters()\n    optimizer = torch.optim.AdamW((p for p in model.parameters() if p.requires_grad),\n                                 lr=args.learning_rate, weight_decay=0.01)\n    scheduler = get_cosine_schedule_with_warmup(optimizer, round(args.steps * args.warmup_ratio), args.steps)\n    scaler = torch.amp.GradScaler(\'cuda\', enabled=not bf16)\n    counts = manifest[\'train_groups\']\n    if args.mode != \'uniform\' and len(counts) < 2:\n        raise ValueError(\'Adaptive/group balancing requires at least two train groups; inspect grouping metadata\')\n    controller = GroupWeights(sorted(counts), counts, args.mode, args.mix,\n                              args.temperature, args.ema, args.weight_floor, args.weight_ceiling)\n    state = {\'step\': 0, \'epoch\': 0, \'offset\': 0, \'train_tokens\': 0, \'response_tokens\': 0,\n             \'probe_tokens\': 0, \'holdout_tokens\': 0, \'elapsed_seconds\': 0., \'probe_seconds\': 0.,\n             \'holdout_seconds\': 0., \'optimizer_updates\': 0, \'peak_vram_bytes\': 0}\n    if checkpoint:\n        # Only load a checkpoint produced by this script from your own run.\n        saved = torch.load(checkpoint / \'state.pt\', map_location=\'cpu\', weights_only=False)\n        optimizer.load_state_dict(saved[\'optimizer\'])\n        scheduler.load_state_dict(saved[\'scheduler\'])\n        scaler.load_state_dict(saved[\'scaler\'])\n        controller.__dict__.update(saved[\'controller\'])\n        state = saved[\'state\']\n        random.setstate(saved[\'python_rng\'])\n        np.random.set_state(saved[\'numpy_rng\'])\n        torch.set_rng_state(saved[\'torch_rng\'])\n        torch.cuda.set_rng_state_all(saved[\'cuda_rng\'])\n        if state[\'step\'] >= args.steps:\n            raise ValueError(\'Checkpoint already reached requested steps\')\n        # Remove stale post-checkpoint log entries after a crash, without duplicating steps.\n        log = out / \'metrics.jsonl\'\n        if log.exists():\n            lines = [json.loads(s) for s in log.read_text().splitlines() if s.strip()]\n            log.write_text(\'\'.join(json.dumps(r) + \'\\n\' for r in lines if r[\'step\'] <= state[\'step\']))\n    start = time.monotonic()\n    prior_elapsed = state[\'elapsed_seconds\']\n    torch.cuda.reset_peak_memory_stats()\n\n    def batch(rows):\n        length = max(len(r[\'input_ids\']) for r in rows)\n        ids = [r[\'input_ids\'] + [tokenizer.pad_token_id] * (length - len(r[\'input_ids\'])) for r in rows]\n        labels = [r[\'labels\'] + [-100] * (length - len(r[\'labels\'])) for r in rows]\n        mask = [[1] * len(r[\'input_ids\']) + [0] * (length - len(r[\'input_ids\'])) for r in rows]\n        return {k: torch.tensor(v, device=\'cuda\') for k, v in\n                [(\'input_ids\', ids), (\'attention_mask\', mask), (\'labels\', labels)]}\n\n    def measure(split):\n        torch.cuda.synchronize()\n        begin = time.monotonic()\n        totals, sizes = defaultdict(float), Counter()\n        model.eval()\n        with torch.inference_mode(), torch.autocast(\'cuda\', dtype=dtype):\n            for r in data[split]:\n                b = batch([r])\n                labels = b.pop(\'labels\')\n                output = model(**b, use_cache=False)\n                loss = per_example_loss(output.logits, labels).item()\n                totals[r[\'train_group\']] += loss\n                sizes[r[\'train_group\']] += 1\n                state[f\'{split}_tokens\'] += len(r[\'input_ids\'])\n                del output, labels, b\n        model.train()\n        import gc\n        gc.collect()\n        torch.cuda.empty_cache()\n        torch.cuda.synchronize()\n        state[f\'{split}_seconds\'] += time.monotonic() - begin\n        return {g: totals[g] / sizes[g] for g in totals}\n\n    def log(value):\n        with (out / \'metrics.jsonl\').open(\'a\') as stream:\n            stream.write(json.dumps({\'step\': state[\'step\'], **value}) + \'\\n\')\n        print(json.dumps({\'step\': state[\'step\'], **value}), flush=True)\n\n    def update_controller(losses):\n        # Shared dev contains all groups even when a selector drops some from\n        # train. Report all losses, but only train groups have controller weights.\n        supported = {g: value for g, value in losses.items() if g in counts}\n        if supported:\n            controller.update(supported)\n\n    def save():\n        torch.cuda.synchronize()\n        state[\'elapsed_seconds\'] = prior_elapsed + time.monotonic() - start\n        state[\'peak_vram_bytes\'] = max(state[\'peak_vram_bytes\'], torch.cuda.max_memory_allocated())\n        target = out / f"checkpoint-{state[\'step\']}"\n        temp = out / f"checkpoint-{state[\'step\']}.tmp"\n        if target.exists() or temp.exists():\n            raise ValueError(f\'Checkpoint path exists: {target}; preserve or move it before resuming\')\n        temp.mkdir()\n        model.save_pretrained(temp / \'adapter\')\n        tokenizer.save_pretrained(temp / \'adapter\')\n        write_json(temp / \'identity.json\', identity)\n        torch.save({\'optimizer\': optimizer.state_dict(), \'scheduler\': scheduler.state_dict(),\n                    \'scaler\': scaler.state_dict(), \'controller\': controller.__dict__, \'state\': state,\n                    \'python_rng\': random.getstate(), \'numpy_rng\': np.random.get_state(),\n                    \'torch_rng\': torch.get_rng_state(), \'cuda_rng\': torch.cuda.get_rng_state_all()}, temp / \'state.pt\')\n        temp.rename(target)\n        write_json(out / \'latest.json\', {\'checkpoint\': str(target.resolve()), \'state\': state})\n\n    if not checkpoint:\n        initial = measure(\'probe\')\n        update_controller(initial)\n        log({\'event\': \'initial_probe\', \'probe_loss\': initial, \'weights\': controller.weights,\n             \'holdout_loss\': measure(\'holdout\')})\n    model.train()\n    order_epoch, order = None, None\n    while state[\'step\'] < args.steps:\n        optimizer.zero_grad(set_to_none=True)\n        micro_losses = []\n        for _ in range(args.grad_accum):\n            rows = []\n            while len(rows) < args.batch_size:\n                if order_epoch != state[\'epoch\']:\n                    order_epoch = state[\'epoch\']\n                    order = list(range(len(data[\'train\'])))\n                    random.Random(args.seed + state[\'epoch\']).shuffle(order)\n                take = min(args.batch_size - len(rows), len(order) - state[\'offset\'])\n                rows.extend(data[\'train\'][i] for i in order[state[\'offset\']:state[\'offset\'] + take])\n                state[\'offset\'] += take\n                if state[\'offset\'] == len(order):\n                    state[\'epoch\'] += 1\n                    state[\'offset\'] = 0\n            b = batch(rows)\n            labels = b.pop(\'labels\')\n            with torch.autocast(\'cuda\', dtype=dtype):\n                output = model(**b, use_cache=False)\n                losses = per_example_loss(output.logits, labels)\n                weights = losses.new_tensor([controller.weights[r[\'train_group\']] for r in rows])\n                loss = (losses * weights).mean() / args.grad_accum\n            if not torch.isfinite(loss):\n                raise FloatingPointError(\'Non-finite loss; stop and lower LR/check data/precision\')\n            scaler.scale(loss).backward()\n            micro_losses.append(losses.detach().mean().item())\n            state[\'train_tokens\'] += sum(len(r[\'input_ids\']) for r in rows)\n            state[\'response_tokens\'] += sum(r[\'response_tokens\'] for r in rows)\n            del output, losses, loss, weights, labels, b\n        scaler.unscale_(optimizer)\n        grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), 1.)\n        old_scale = scaler.get_scale()\n        scaler.step(optimizer)\n        scaler.update()\n        if scaler.get_scale() >= old_scale:\n            scheduler.step()\n            state[\'optimizer_updates\'] += 1\n        state[\'step\'] += 1\n        if state[\'step\'] % args.log_every == 0 or state[\'step\'] == args.steps:\n            import gc\n            gc.collect()\n            torch.cuda.empty_cache()\n            vram_alloc = torch.cuda.memory_allocated() / (1024**3)\n            vram_res = torch.cuda.memory_reserved() / (1024**3)\n            log({\'event\': \'train\', \'response_ce\': sum(micro_losses) / len(micro_losses),\n                 \'grad_norm\': float(grad_norm), \'learning_rate\': scheduler.get_last_lr()[0],\n                 \'train_tokens\': state[\'train_tokens\'], \'response_tokens\': state[\'response_tokens\'],\n                 \'vram_alloc_gb\': round(vram_alloc, 2), \'vram_reserved_gb\': round(vram_res, 2),\n                 \'optimizer_updates\': state[\'optimizer_updates\']})\n        if state[\'step\'] % args.probe_every == 0 or state[\'step\'] == args.steps:\n            losses = measure(\'probe\')\n            if state[\'step\'] >= args.warmup_steps:\n                update_controller(losses)\n            log({\'event\': \'probe\', \'probe_loss\': losses, \'weights\': controller.weights,\n                 \'holdout_loss\': measure(\'holdout\')})\n        if state[\'step\'] % args.save_every == 0 or state[\'step\'] == args.steps:\n            save()\n        if args.max_seconds and time.monotonic() - start >= args.max_seconds and state[\'step\'] < args.steps:\n            if state[\'step\'] % args.save_every:\n                save()\n            write_json(out / \'summary.json\', {\'completed\': False, \'mode\': args.mode, \'model\': manifest[\'model\'],\n                \'state\': state, \'weights\': controller.weights, \'note\': \'Time budget reached at accumulation boundary; resume latest checkpoint.\'})\n            print(\'Stage paused at an optimizer boundary. Restore latest full checkpoint to resume.\', flush=True)\n            return\n    model.save_pretrained(out / \'adapter\')\n    tokenizer.save_pretrained(out / \'adapter\')\n    write_json(out / \'summary.json\', {\'completed\': True, \'mode\': args.mode, \'model\': manifest[\'model\'], \'state\': state,\n                                    \'weights\': controller.weights,\n                                    \'note\': \'Teacher-response CE is not benchmark accuracy/F1. Run SentiBench separately.\'})\n\n\ndef export(args):\n    import torch\n    _disable_torchao()\n    from peft import PeftModel\n    from transformers import AutoModelForCausalLM, AutoTokenizer\n    run = Path(args.run)\n    identity = json.loads((run / \'identity.json\').read_text())\n    prepared = Path(identity[\'config\'][\'prepared\'])\n    manifest = json.loads((prepared / \'manifest.json\').read_text())\n    target = Path(args.output)\n    if target.exists() and any(target.iterdir()):\n        raise ValueError(\'Export target must be empty\')\n    model = AutoModelForCausalLM.from_pretrained(manifest[\'model\'], revision=manifest[\'model_revision\'],\n                                               torch_dtype=torch.float32, device_map={\'\': \'cpu\'})\n    model = PeftModel.from_pretrained(model, run / \'adapter\').merge_and_unload()\n    model.save_pretrained(target, safe_serialization=True)\n    AutoTokenizer.from_pretrained(run / \'adapter\').save_pretrained(target)\n    write_json(target / \'distillation_run.json\', identity)\n    print(\'Merged model:\', target)\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    commands = parser.add_subparsers(dest=\'command\', required=True)\n    a = commands.add_parser(\'audit\')\n    a.add_argument(\'--file\', required=True)\n    a.add_argument(\'--group_field\', default=\'cluster_id\')\n    a.add_argument(\'--examples\', type=int, default=5)\n    a.add_argument(\'--output\', required=True)\n    a.set_defaults(func=audit)\n    p = commands.add_parser(\'prepare\')\n    p.add_argument(\'--file\', required=True)\n    p.add_argument(\'--output\', required=True)\n    p.add_argument(\'--model\', default=\'Qwen/Qwen3-0.6B\')\n    p.add_argument(\'--model_revision\', default=\'main\')\n    p.add_argument(\'--corpus_revision\', required=True)\n    p.add_argument(\'--prompt_fields\', required=True, help=\'Comma-separated fields in verified order\')\n    p.add_argument(\'--response_field\', required=True)\n    p.add_argument(\'--group_field\', required=True)\n    p.add_argument(\'--source_field\', required=True, help=\'Original input text for leakage-safe grouped splitting\')\n    p.add_argument(\'--mapping_confirmed\', action=\'store_true\')\n    p.add_argument(\'--max_length\', type=int, default=2048)\n    p.add_argument(\'--max_train\', type=int, default=5000, help=\'0 = all eligible training records\')\n    p.add_argument(\'--seed\', type=int, default=42)\n    p.add_argument(\'--dev_fraction\', type=float, default=0.1)\n    p.add_argument(\'--probe_per_group\', type=int, default=4)\n    p.add_argument(\'--holdout_per_group\', type=int, default=4)\n    p.add_argument(\'--min_group_train\', type=int, default=8)\n    p.add_argument(\'--max_groups\', type=int, default=16)\n    p.set_defaults(func=prepare)\n    t = commands.add_parser(\'train\')\n    t.add_argument(\'--prepared\', required=True)\n    t.add_argument(\'--output\', required=True)\n    t.add_argument(\'--mode\', choices=[\'uniform\', \'balanced\', \'loss\', \'progress\'], default=\'uniform\')\n    t.add_argument(\'--steps\', type=int, default=200)\n    t.add_argument(\'--batch_size\', type=int, default=1)\n    t.add_argument(\'--grad_accum\', type=int, default=8)\n    t.add_argument(\'--learning_rate\', type=float, default=1e-4)\n    t.add_argument(\'--rank\', type=int, default=16)\n    t.add_argument(\'--target_modules\', default=\'q_proj,k_proj,v_proj,o_proj\',\n                   help=\'Comma-separated LoRA target module names\')\n    t.add_argument(\'--warmup_ratio\', type=float, default=0.05)\n    t.add_argument(\'--warmup_steps\', type=int, default=25, help=\'Before adaptive updates\')\n    t.add_argument(\'--probe_every\', type=int, default=25)\n    t.add_argument(\'--save_every\', type=int, default=100)\n    t.add_argument(\'--log_every\', type=int, default=10)\n    t.add_argument(\'--mix\', type=float, default=0.5)\n    t.add_argument(\'--temperature\', type=float, default=0.05)\n    t.add_argument(\'--ema\', type=float, default=0.5)\n    t.add_argument(\'--weight_floor\', type=float, default=0.5)\n    t.add_argument(\'--weight_ceiling\', type=float, default=2.)\n    t.add_argument(\'--seed\', type=int, default=42)\n    t.add_argument(\'--four_bit\', action=\'store_true\')\n    t.add_argument(\'--resume\', help=\'Own checkpoint directory; restores optimizer/scaler/RNG/controller/cursor\')\n    t.add_argument(\'--init_adapter\', help=\'Own previous-stage adapter; new optimizer/scheduler, unlike resume\')\n    t.add_argument(\'--max_seconds\', type=float, default=0., help=\'0: unlimited; save and stop at optimizer boundary\')\n    t.set_defaults(func=train)\n    e = commands.add_parser(\'export\')\n    e.add_argument(\'--run\', required=True)\n    e.add_argument(\'--output\', required=True)\n    e.set_defaults(func=export)\n    args = parser.parse_args()\n    args.func(args)\n\n\nif __name__ == \'__main__\':\n    main()\n', 'training/prepare_v1_pruned.py': '"""Kaggle v1 A/B preparation. No heavy imports until tokenize/score/download.\n\nPlan A: coverage + base-student conditional/unconditional response-loss ratio.\nPlan B: coverage only, available without model-scoring cost.\nBoth plans share random control, ICLDIST IDs and development splits.\n"""\nimport argparse\nimport os\nos.environ.setdefault(\'PYTORCH_CUDA_ALLOC_CONF\', \'expandable_segments:True\')\nfrom collections import Counter\nfrom importlib.metadata import version\nimport json\nimport math\nfrom pathlib import Path\nimport shutil\nimport time\n\nfrom distill_core import digest, file_hash, read_records, write_json\nfrom v1_core import (canonical_record, iter_objects, limited_candidates, normalized_text,\n                     select_rows, split_name)\n\n\ndef fresh(path):\n    path = Path(path)\n    if path.exists() and any(path.iterdir()):\n        raise ValueError(f\'Output is not empty: {path}; choose a new experiment directory\')\n    path.mkdir(parents=True, exist_ok=True)\n    return path\n\n\ndef jsonl(path, rows):\n    with Path(path).open(\'w\', encoding=\'utf-8\') as stream:\n        for row in rows:\n            stream.write(json.dumps(row, ensure_ascii=False) + \'\\n\')\n\n\ndef download(args):\n    from huggingface_hub import HfApi, snapshot_download\n    revision = HfApi().dataset_info(\'zhang-yice/sentiment-distillation-v1\', revision=args.revision).sha\n    directory = snapshot_download(\'zhang-yice/sentiment-distillation-v1\', repo_type=\'dataset\',\n                                  revision=revision, local_dir=args.output,\n                                  allow_patterns=[\'KNOWDIST_200k/**\', \'ICLDIST_100k/**\'])\n    write_json(Path(args.output) / \'download-manifest.json\', {\'dataset\': \'zhang-yice/sentiment-distillation-v1\',\n                                                            \'revision\': revision, \'directory\': directory})\n    print(f\'Downloaded v1 revision {revision}; inspect schemas before normalize.\')\n\n\ndef normalize(args):\n    if not args.mapping_confirmed:\n        raise ValueError(\'Inspect full source records/extraction rules; then pass --mapping-confirmed\')\n    root, out = Path(args.input), fresh(args.output)\n    benchmark_keys, benchmark_files = set(), {}\n    if args.benchmark_root:\n        files = sorted(Path(args.benchmark_root).glob(\'*/*/test.json\'))\n        if not files:\n            raise ValueError(\'No benchmark */*/test.json files found\')\n        for path in files:\n            benchmark_files[str(path.relative_to(args.benchmark_root))] = file_hash(path)\n            for _, obj in iter_objects(path):\n                text = next((obj[k] for k in (\'sentence\', \'text\', \'review\')\n                             if isinstance(obj.get(k), str) and obj[k].strip()), None)\n                if text is None:\n                    raise ValueError(f\'Unsupported benchmark text schema: {path}; audit required\')\n                benchmark_keys.add(digest(normalized_text(text)))\n    files = sorted(p for p in root.rglob(\'*\') if p.suffix in (\'.json\', \'.jsonl\')\n                   and not p.name.endswith(\'manifest.json\') and \'.cache\' not in p.parts)\n    if not files:\n        raise ValueError(\'No JSON/JSONL corpus files found\')\n    counts, seen, hashes = Counter(), set(), {}\n    with (out / \'canonical.jsonl\').open(\'w\', encoding=\'utf-8\') as stream, (out / \'exclusions.jsonl\').open(\'w\', encoding=\'utf-8\') as exclusions:\n        for path in files:\n            name, sha = str(path.relative_to(root)), file_hash(path)\n            hashes[name] = sha\n            for index, obj in iter_objects(path):\n                try:\n                    r = canonical_record(obj, name, index, sha)\n                    key = digest([r[\'stage\'], r[\'prompt\'], r[\'response\']])\n                    if key in seen:\n                        raise ValueError(\'duplicate_prompt_response\')\n                    if r[\'source_key\'] in benchmark_keys:\n                        raise ValueError(\'exact_normalized_benchmark_test_overlap\')\n                    seen.add(key)\n                    r[\'split\'] = split_name(r[\'source_key\'], args.seed, args.dev_fraction)\n                    stream.write(json.dumps(r, ensure_ascii=False) + \'\\n\')\n                    counts[f\'{r["stage"]}/{r["split"]}/{r["group"]}\'] += 1\n                except ValueError as exc:\n                    exclusions.write(json.dumps({\'file\': name, \'position\': index, \'id\': f\'{sha}:{index}\',\n                                                 \'reason\': str(exc)}, ensure_ascii=False) + \'\\n\')\n                    counts[\'excluded\'] += 1\n    if not any(k.startswith(\'knowdist/train/\') for k in counts) or not any(k.startswith(\'icldist/train/\') for k in counts):\n        raise ValueError(\'Missing train stage; inspect exclusions and mappings\')\n    write_json(out / \'manifest.json\', {\'protocol\': \'v1-ab-r1\', \'corpus_revision\': args.revision,\n                \'files\': hashes, \'counts\': dict(counts), \'seed\': args.seed, \'dev_fraction\': args.dev_fraction,\n                \'source_extractor\': \'final_input_marker_v1\',\n                \'source_sha256\': digest([file_hash(Path(__file__)), file_hash(Path(__file__).with_name(\'v1_core.py\'))]),\n                \'canonical_sha256\': file_hash(out / \'canonical.jsonl\'), \'benchmark_files\': benchmark_files,\n                \'limitations\': [\'Exact normalized final-input overlap only; no near-duplicate/substring audit.\',\n                               \'Embedded demonstrations are not isolated across splits; audit needed before generalization claims.\',\n                               \'Unsupported source templates are excluded; inspect coverage and exclusion rates.\']})\n    print(json.dumps(dict(counts), indent=2))\n\n\ndef tokenize(args):\n    from huggingface_hub import HfApi\n    from transformers import AutoTokenizer\n    source, out = Path(args.normalized), fresh(args.output)\n    manifest = json.loads((source / \'manifest.json\').read_text())\n    if file_hash(source / \'canonical.jsonl\') != manifest[\'canonical_sha256\']:\n        raise ValueError(\'Canonical source changed\')\n    revision = HfApi().model_info(args.model, revision=args.model_revision).sha\n    tokenizer = AutoTokenizer.from_pretrained(args.model, revision=revision)\n    if tokenizer.eos_token_id is None or not tokenizer.chat_template:\n        raise ValueError(\'Chat template and EOS required\')\n    if args.max_length < 3:\n        raise ValueError(\'max_length too small\')\n    candidates = limited_candidates((r for _, r in read_records(source / \'canonical.jsonl\')),\n            args.seed, args.per_group, {\'knowdist\': args.know_candidates, \'icldist\': args.icl_candidates})\n    retained, excluded = [], []\n    for r in candidates:\n        prefix = tokenizer.apply_chat_template([{\'role\': \'user\', \'content\': r[\'prompt\']}],\n                   tokenize=False, add_generation_prompt=True, enable_thinking=False)\n        prompt_ids = tokenizer.encode(prefix, add_special_tokens=False)\n        answer = tokenizer.encode(r[\'response\'], add_special_tokens=False)\n        if not prompt_ids or not answer:\n            excluded.append({\'id\': r[\'id\'], \'reason\': \'empty_tokenized_prompt_or_response\'})\n            continue\n        if answer[-1] != tokenizer.eos_token_id:\n            answer.append(tokenizer.eos_token_id)\n        if len(prompt_ids) + len(answer) > args.max_length:\n            excluded.append({\'id\': r[\'id\'], \'reason\': \'whole_record_overlength\'})\n            continue\n        # Drop raw source_text from tokenized pool; final prompt/response remain unchanged.\n        r = {k: v for k, v in r.items() if k != \'source_text\'}\n        r.update(input_ids=prompt_ids + answer, labels=[-100] * len(prompt_ids) + answer,\n                 prompt_tokens=len(prompt_ids), response_tokens=len(answer), train_group=r[\'group\'])\n        retained.append(r)\n    for stage in (\'knowdist\', \'icldist\'):\n        for split in (\'train\', \'probe\', \'holdout\'):\n            if not any(r[\'stage\'] == stage and r[\'split\'] == split for r in retained):\n                raise ValueError(f\'Empty {stage}/{split}; expand corpus/candidate/dev limits\')\n    tokenizer.save_pretrained(out / \'tokenizer\')\n    jsonl(out / \'pool.jsonl\', retained)\n    jsonl(out / \'exclusions.jsonl\', excluded)\n    write_json(out / \'manifest.json\', {\'protocol\': \'v1-ab-r1\', \'model\': args.model, \'model_revision\': revision,\n        \'corpus_revision\': manifest[\'corpus_revision\'], \'normalization_manifest_sha256\': file_hash(source / \'manifest.json\'),\n        \'pool_sha256\': file_hash(out / \'pool.jsonl\'), \'seed\': args.seed, \'max_length\': args.max_length,\n        \'counts\': dict(Counter(f\'{r["stage"]}/{r["split"]}/{r["group"]}\' for r in retained)),\n        \'candidate_caps\': {\'knowdist\': args.know_candidates, \'icldist\': args.icl_candidates, \'per_group\': args.per_group},\n        \'excluded\': len(excluded), \'tokenizer_template_sha256\': digest(tokenizer.chat_template),\n        \'tokenizer_hashes\': {p.name: file_hash(p) for p in sorted((out / \'tokenizer\').iterdir()) if p.is_file()}})\n    write_json(out / \'preview.json\', [{k: r[k] for k in (\'id\', \'stage\', \'group\', \'split\', \'prompt\', \'response\')}\n                                     for r in retained[:16]])\n    print(f\'Retained {len(retained)} candidates; excluded {len(excluded)}. Inspect preview and coverage.\')\n\n\ndef load_pool(directory):\n    directory = Path(directory)\n    manifest = json.loads((directory / \'manifest.json\').read_text())\n    if file_hash(directory / \'pool.jsonl\') != manifest[\'pool_sha256\']:\n        raise ValueError(\'Tokenized pool changed\')\n    for name, sha in manifest[\'tokenizer_hashes\'].items():\n        if file_hash(directory / \'tokenizer\' / name) != sha:\n            raise ValueError(\'Tokenizer changed\')\n    return manifest, [r for _, r in read_records(directory / \'pool.jsonl\')]\n\n\ndef score(args):\n    """Base-student IFD-style proxy; no claim of reproducing pre-experience IFD."""\n    import torch\n    from transformers import AutoModelForCausalLM, AutoTokenizer\n    from train_distill import per_example_loss\n    manifest, pool = load_pool(args.pool)\n    if not torch.cuda.is_available():\n        raise ValueError(\'Score on Kaggle CUDA; local model execution is not part of this workflow\')\n    rows = [r for r in pool if r[\'stage\'] == \'knowdist\' and r[\'split\'] == \'train\']\n    if args.max_seconds < 0:\n        raise ValueError(\'max-seconds must be nonnegative\')\n    dtype = torch.bfloat16 if torch.cuda.get_device_capability(0)[0] >= 8 and torch.cuda.is_bf16_supported() else torch.float16\n    identity = {\'pool_sha256\': manifest[\'pool_sha256\'], \'model\': manifest[\'model\'],\n        \'model_revision\': manifest[\'model_revision\'], \'tokenizer_hashes\': manifest[\'tokenizer_hashes\'],\n        \'recipe\': \'base_student_chat_response_loss_ratio_v1\', \'dtype\': str(dtype),\n        \'source_sha256\': digest([file_hash(Path(__file__)), file_hash(Path(__file__).with_name(\'train_distill.py\'))]),\n        \'versions\': {p: version(p) for p in (\'torch\', \'transformers\')}, \'gpu\': torch.cuda.get_device_name(0)}\n    out = Path(args.output)\n    scores, prior_elapsed = {}, 0.\n    if args.resume:\n        saved = json.loads((out / \'identity.json\').read_text())\n        if saved != identity:\n            raise ValueError(\'Score resume identity differs\')\n        for _, r in (read_records(out / \'scores.jsonl\') if (out / \'scores.jsonl\').exists() else []):\n            if r[\'id\'] in scores:\n                raise ValueError(\'Duplicate score ID\')\n            scores[r[\'id\']] = r\n        if (out / \'summary.json\').exists():\n            prior_elapsed = json.loads((out / \'summary.json\').read_text())[\'elapsed_seconds\']\n        prior_elapsed = max(prior_elapsed, max((r.get(\'elapsed_seconds\', 0.) for r in scores.values()), default=0.))\n    else:\n        out = fresh(out)\n        write_json(out / \'identity.json\', identity)\n    begin = time.monotonic()\n    tokenizer = AutoTokenizer.from_pretrained(Path(args.pool) / \'tokenizer\')\n    bos = tokenizer.bos_token_id if tokenizer.bos_token_id is not None else tokenizer.eos_token_id\n    model = AutoModelForCausalLM.from_pretrained(manifest[\'model\'], revision=manifest[\'model_revision\'],\n               torch_dtype=dtype, device_map={\'\': 0}, attn_implementation=\'sdpa\')\n    model.eval()\n    completed = True\n    with (out / \'scores.jsonl\').open(\'a\', encoding=\'utf-8\') as stream, torch.inference_mode():\n        for r in rows:\n            if r[\'id\'] in scores:\n                continue\n            ids = torch.tensor([r[\'input_ids\']], device=\'cuda\')\n            labels = torch.tensor([r[\'labels\']], device=\'cuda\')\n            output = model(input_ids=ids, attention_mask=torch.ones_like(ids), use_cache=False)\n            conditioned = float(per_example_loss(output.logits, labels)[0])\n            del output, ids, labels\n            answer = r[\'input_ids\'][r[\'prompt_tokens\']:]\n            ids = torch.tensor([[bos] + answer], device=\'cuda\')\n            labels = torch.tensor([[-100] + answer], device=\'cuda\')\n            output = model(input_ids=ids, attention_mask=torch.ones_like(ids), use_cache=False)\n            unconditioned = float(per_example_loss(output.logits, labels)[0])\n            del output, ids, labels\n            if not all(math.isfinite(x) and x > 0 for x in (conditioned, unconditioned)):\n                raise ValueError(f\'Invalid response losses: {r["id"]}\')\n            record = {\'id\': r[\'id\'], \'conditioned_ce\': conditioned, \'unconditioned_ce\': unconditioned,\n                      \'ifd_proxy\': conditioned / unconditioned,\n                      \'elapsed_seconds\': prior_elapsed + time.monotonic() - begin}\n            stream.write(json.dumps(record) + \'\\n\')\n            stream.flush()\n            scores[r[\'id\']] = record\n            if args.max_seconds and time.monotonic() - begin >= args.max_seconds:\n                completed = len(scores) == len(rows)\n                break\n    torch.cuda.synchronize()\n    write_json(out / \'summary.json\', {\'completed\': completed, \'records\': len(scores), \'expected\': len(rows),\n        \'elapsed_seconds\': prior_elapsed + time.monotonic() - begin,\n        \'identity_sha256\': file_hash(out / \'identity.json\'), \'scores_sha256\': file_hash(out / \'scores.jsonl\')})\n    print(f\'Scored {len(scores)}/{len(rows)}. Partial scores cannot select A; B remains available.\')\n\n\ndef select(args):\n    manifest, pool = load_pool(args.pool)\n    scores = None\n    score_hash = None\n    if args.method == \'student\':\n        if not args.scores:\n            raise ValueError(\'Plan A requires --scores; use coverage explicitly for Plan B\')\n        directory = Path(args.scores)\n        identity = json.loads((directory / \'identity.json\').read_text())\n        summary = json.loads((directory / \'summary.json\').read_text())\n        if not summary[\'completed\'] or identity[\'pool_sha256\'] != manifest[\'pool_sha256\']:\n            raise ValueError(\'Incomplete scores or wrong pool; no silent fallback\')\n        if identity[\'model\'] != manifest[\'model\'] or identity[\'model_revision\'] != manifest[\'model_revision\']:\n            raise ValueError(\'Scores belong to another model\')\n        score_hash = file_hash(directory / \'scores.jsonl\')\n        if score_hash != summary[\'scores_sha256\'] or file_hash(directory / \'identity.json\') != summary[\'identity_sha256\']:\n            raise ValueError(\'Score artifacts changed\')\n        scored_rows = [r for _, r in read_records(directory / \'scores.jsonl\')]\n        scores = {r[\'id\']: r[\'ifd_proxy\'] for r in scored_rows}\n        expected_ids = {r[\'id\'] for r in pool if r[\'stage\'] == \'knowdist\' and r[\'split\'] == \'train\'}\n        if len(scores) != len(scored_rows) or set(scores) != expected_ids:\n            raise ValueError(\'Score IDs do not exactly match KNOWDIST train candidates\')\n    out = fresh(args.output)\n    reports = {}\n    for stage, budget, cap in [(\'knowdist\', args.know_tokens, args.know_records),\n                               (\'icldist\', args.icl_tokens, args.icl_records)]:\n        candidates = [r for r in pool if r[\'stage\'] == stage and r[\'split\'] == \'train\']\n        # Fixed ICLDIST selection in ALL methods, including the random control.\n        method = args.method if stage == \'knowdist\' else \'coverage\'\n        train, report = select_rows(candidates, method, budget, cap, args.seed,\n                                   scores, args.min_ifd, args.max_ifd)\n        if method == \'student\' and report[\'missing_groups\'] and not args.allow_missing_groups:\n            raise ValueError(f\'Plan A lost groups: {report["missing_groups"]}; inspect interval or run B explicitly\')\n        reports[stage] = report\n        stage_out = out / stage\n        stage_out.mkdir()\n        shutil.copytree(Path(args.pool) / \'tokenizer\', stage_out / \'tokenizer\')\n        splits = {\'train\': train}\n        splits.update({s: [r for r in pool if r[\'stage\'] == stage and r[\'split\'] == s]\n                       for s in (\'probe\', \'holdout\')})\n        for name, rows in splits.items():\n            if not rows:\n                raise ValueError(f\'Empty {stage}/{name}\')\n            jsonl(stage_out / f\'{name}.jsonl\', rows)\n        prepared = {k: manifest[k] for k in (\'model\', \'model_revision\', \'corpus_revision\',\n                                           \'tokenizer_hashes\', \'tokenizer_template_sha256\')}\n        prepared.update(protocol=\'v1-ab-r1\', stage=stage, selection=report,\n            pool_sha256=manifest[\'pool_sha256\'], config={\'seed\': args.seed, \'method\': args.method,\n            \'score_hash\': score_hash, \'ifd_interval\': [args.min_ifd, args.max_ifd]},\n            split_hashes={s: file_hash(stage_out / f\'{s}.jsonl\') for s in splits},\n            counts={s: len(rows) for s, rows in splits.items()},\n            train_groups=dict(Counter(r[\'train_group\'] for r in train)),\n            train_input_tokens=sum(len(r[\'input_ids\']) for r in train),\n            train_response_tokens=sum(r[\'response_tokens\'] for r in train))\n        write_json(stage_out / \'manifest.json\', prepared)\n        write_json(stage_out / \'selected_ids.json\', [r[\'id\'] for r in train])\n    write_json(out / \'selection.json\', {\'method\': args.method, \'seed\': args.seed, \'stages\': reports,\n                                      \'pool_sha256\': manifest[\'pool_sha256\'], \'scores_sha256\': score_hash})\n    print(json.dumps(reports, indent=2))\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    commands = parser.add_subparsers(dest=\'command\', required=True)\n    p = commands.add_parser(\'download\')\n    p.add_argument(\'--revision\', required=True)\n    p.add_argument(\'--output\', required=True)\n    p.set_defaults(func=download)\n    p = commands.add_parser(\'normalize\')\n    p.add_argument(\'--input\', required=True)\n    p.add_argument(\'--output\', required=True)\n    p.add_argument(\'--revision\', required=True)\n    p.add_argument(\'--benchmark-root\')\n    p.add_argument(\'--mapping-confirmed\', action=\'store_true\')\n    p.add_argument(\'--seed\', type=int, default=42)\n    p.add_argument(\'--dev-fraction\', type=float, default=.1)\n    p.set_defaults(func=normalize)\n    p = commands.add_parser(\'tokenize\')\n    p.add_argument(\'--normalized\', required=True)\n    p.add_argument(\'--output\', required=True)\n    p.add_argument(\'--model\', default=\'Qwen/Qwen2.5-0.5B-Instruct\')\n    p.add_argument(\'--model-revision\', default=\'main\')\n    p.add_argument(\'--max-length\', type=int, default=2048)\n    p.add_argument(\'--know-candidates\', type=int, default=10000)\n    p.add_argument(\'--icl-candidates\', type=int, default=5000)\n    p.add_argument(\'--per-group\', type=int, default=1000)\n    p.add_argument(\'--seed\', type=int, default=42)\n    p.set_defaults(func=tokenize)\n    p = commands.add_parser(\'score\')\n    p.add_argument(\'--pool\', required=True)\n    p.add_argument(\'--output\', required=True)\n    p.add_argument(\'--resume\', action=\'store_true\')\n    p.add_argument(\'--max-seconds\', type=float, default=7200)\n    p.set_defaults(func=score)\n    p = commands.add_parser(\'select\')\n    p.add_argument(\'--pool\', required=True)\n    p.add_argument(\'--output\', required=True)\n    p.add_argument(\'--method\', choices=[\'random\', \'coverage\', \'student\'], required=True)\n    p.add_argument(\'--scores\')\n    p.add_argument(\'--min-ifd\', type=float, default=0.)\n    p.add_argument(\'--max-ifd\', type=float, default=1.)\n    p.add_argument(\'--allow-missing-groups\', action=\'store_true\', help=\'Explicitly accept/report loss of group coverage in A\')\n    p.add_argument(\'--know-tokens\', type=int, default=1500000)\n    p.add_argument(\'--icl-tokens\', type=int, default=500000)\n    p.add_argument(\'--know-records\', type=int, default=3500)\n    p.add_argument(\'--icl-records\', type=int, default=1500)\n    p.add_argument(\'--seed\', type=int, default=42)\n    p.set_defaults(func=select)\n    args = parser.parse_args()\n    args.func(args)\n\n\nif __name__ == \'__main__\':\n    main()\n', 'training/run_v1_ab.py': '"""Run interchangeable v1 plans with immutable config and no test-driven switch.\n\nPlan directories remain separate. Resuming a partial stage restores full state;\nstarting ICLDIST loads the completed KNOWDIST adapter with a new optimizer.\n"""\nimport argparse\nimport json\nfrom pathlib import Path\nimport subprocess\nimport sys\nimport time\n\nfrom distill_core import digest, file_hash, read_records, write_json\nfrom v1_core import steps_for_tokens\n\n\ndef build_request(prepared, config, plan, seed, pilot_steps=0):\n    prepared = Path(prepared)\n    stages, shared_model, model_revision, tokenizer_hashes, pool_sha = {}, None, None, None, None\n    for stage in (\'knowdist\', \'icldist\'):\n        directory = prepared / stage\n        manifest = json.loads((directory / \'manifest.json\').read_text())\n        for split, sha in manifest[\'split_hashes\'].items():\n            if file_hash(directory / f\'{split}.jsonl\') != sha:\n                raise ValueError(f\'Prepared split changed: {stage}/{split}\')\n        if shared_model is not None and (manifest[\'model\'], manifest[\'model_revision\']) != (shared_model, model_revision):\n            raise ValueError(\'Stages use different base model/revision\')\n        if tokenizer_hashes is not None and manifest[\'tokenizer_hashes\'] != tokenizer_hashes:\n            raise ValueError(\'Stages use different tokenizers\')\n        if pool_sha is not None and manifest[\'pool_sha256\'] != pool_sha:\n            raise ValueError(\'Stages use different candidate pools\')\n        expected_method = config[\'plans\'][plan][\'selection\'] if stage == \'knowdist\' else \'coverage\'\n        if manifest[\'selection\'][\'method\'] != expected_method:\n            raise ValueError(\'Stage selection does not match plan\')\n        tokenizer_hashes, pool_sha = manifest[\'tokenizer_hashes\'], manifest[\'pool_sha256\']\n        shared_model, model_revision = manifest[\'model\'], manifest[\'model_revision\']\n        train = [r for _, r in read_records(directory / \'train.jsonl\')]\n        steps, actual = steps_for_tokens(train, config[\'train_tokens\'][stage],\n                                         config[\'batch_size\'], config[\'grad_accum\'], seed)\n        if pilot_steps:\n            steps, actual = pilot_steps, None\n        stages[stage] = {\'prepared\': str(directory.resolve()), \'manifest_sha256\': file_hash(directory / \'manifest.json\'),\n                         \'steps\': steps, \'expected_train_tokens\': actual,\n                         \'target_train_tokens\': config[\'train_tokens\'][stage]}\n    source = {name: file_hash(Path(__file__).with_name(name)) for name in\n              (\'run_v1_ab.py\', \'train_distill.py\', \'distill_core.py\', \'v1_core.py\', \'prepare_v1_pruned.py\')}\n    return {\'config\': config, \'plan\': plan, \'seed\': seed, \'pilot_steps\': pilot_steps,\n            \'source_sha256\': digest(source),\n            \'model\': shared_model, \'model_revision\': model_revision, \'stages\': stages}\n\n\ndef stage_artifacts(directory):\n    files = [directory / \'summary.json\', directory / \'identity.json\']\n    files += sorted(p for p in (directory / \'adapter\').iterdir() if p.is_file())\n    if not any(p.suffix in (\'.safetensors\', \'.bin\') for p in files):\n        raise ValueError(\'Completed stage has no adapter weights\')\n    return {str(p.relative_to(directory)): file_hash(p) for p in files}\n\n\ndef commands(request, output):\n    config, stages = request[\'config\'], request[\'stages\']\n    result = []\n    for stage in (\'knowdist\', \'icldist\'):\n        args = [sys.executable, str(Path(__file__).with_name(\'train_distill.py\')), \'train\',\n            \'--prepared\', stages[stage][\'prepared\'], \'--output\', str(Path(output) / stage),\n            \'--steps\', str(stages[stage][\'steps\']), \'--seed\', str(request[\'seed\']),\n            \'--mode\', config[\'plans\'][request[\'plan\']][\'mode\'], \'--batch_size\', str(config[\'batch_size\']),\n            \'--grad_accum\', str(config[\'grad_accum\']), \'--learning_rate\', str(config[\'learning_rate\']),\n            \'--rank\', str(config[\'rank\']), \'--target_modules\', config[\'target_modules\'],\n            \'--probe_every\', str(config[\'probe_every\']), \'--save_every\', str(config[\'save_every\'])]\n        if stage == \'icldist\':\n            args += [\'--init_adapter\', str(Path(output) / \'knowdist\' / \'adapter\')]\n        result.append((stage, args))\n    return result\n\n\ndef run(args):\n    config = json.loads(Path(args.config).read_text())\n    if args.plan not in config[\'plans\']:\n        raise ValueError(\'Plan not configured\')\n    selection = json.loads((Path(args.prepared) / \'selection.json\').read_text())\n    if selection[\'method\'] != config[\'plans\'][args.plan][\'selection\']:\n        raise ValueError(\'Selected dataset does not match plan; explicit switch requires another prepared directory\')\n    request = build_request(args.prepared, config, args.plan, args.seed, args.pilot_steps)\n    output = Path(args.output).resolve()\n    if args.dry_run:\n        print(json.dumps({\'request\': request, \'commands\': commands(request, output)}, indent=2))\n        return\n    if not 0 < args.max_seconds <= 36000:\n        raise ValueError(\'Set a per-invocation time budget in (0,36000] seconds, below 12h session limit\')\n    output.mkdir(parents=True, exist_ok=True)\n    request_path = output / \'request.json\'\n    if request_path.exists():\n        if json.loads(request_path.read_text()) != request:\n            raise ValueError(\'Run configuration differs; use a new output path\')\n    elif any(output.iterdir()):\n        raise ValueError(\'Nonempty output without request identity\')\n    else:\n        write_json(request_path, request)\n    begin = time.monotonic()\n    for stage, command in commands(request, output):\n        directory = output / stage\n        summary_path = directory / \'summary.json\'\n        if summary_path.exists() and json.loads(summary_path.read_text()).get(\'completed\'):\n            identity = json.loads((directory / \'identity.json\').read_text())\n            if identity[\'manifest_sha256\'] != request[\'stages\'][stage][\'manifest_sha256\']:\n                raise ValueError(\'Completed stage manifest identity differs\')\n            integrity = directory / \'artifact-integrity.json\'\n            if not integrity.exists() or json.loads(integrity.read_text()) != stage_artifacts(directory):\n                raise ValueError(\'Completed stage artifact integrity missing or changed; preserve and inspect\')\n            print(f\'Reusing completed {stage}\')\n            continue\n        if stage == \'icldist\' and not (output / \'knowdist\' / \'adapter\' / \'adapter_config.json\').is_file():\n            raise ValueError(\'KNOWDIST is not complete; cannot start ICLDIST\')\n        remaining = args.max_seconds - (time.monotonic() - begin) - 120\n        if remaining <= 0:\n            print(\'Session budget exhausted; rerun this command after restoring output in next session.\')\n            return\n        command += [\'--max_seconds\', str(remaining)]\n        latest = directory / \'latest.json\'\n        if latest.exists():\n            # Resolve under current writable directory, not an old /kaggle/input mount.\n            name = Path(json.loads(latest.read_text())[\'checkpoint\']).name\n            command += [\'--resume\', str(directory / name)]\n            # Resume retains init_adapter in identity, but train requires it as ancestry,\n            # not as a second initialization. train handles that distinction.\n        elif directory.exists() and any(directory.iterdir()):\n            raise ValueError(f\'Incomplete stage has no full checkpoint: {directory}; preserve and inspect\')\n        subprocess.run(command, check=True)\n        summary = json.loads(summary_path.read_text())\n        if not summary.get(\'completed\'):\n            print(f\'{stage} saved a checkpoint; resume same plan/output before continuing.\')\n            return\n        actual = summary[\'state\'][\'train_tokens\']\n        expected = request[\'stages\'][stage][\'expected_train_tokens\']\n        if expected is not None and actual != expected:\n            raise ValueError(f\'Training token count differs from planned batch stream: {actual} != {expected}\')\n        write_json(directory / \'artifact-integrity.json\', stage_artifacts(directory))\n    write_json(output / \'completed.json\', {\'request_sha256\': file_hash(request_path),\n               \'plan\': args.plan, \'seed\': args.seed,\n               \'stage_summaries\': {s: json.loads((output / s / \'summary.json\').read_text())\n                                   for s in (\'knowdist\', \'icldist\')},\n               \'note\': \'Teacher-response CE only; use an audited benchmark evaluator for F1/Accuracy.\'})\n\n\ndef main():\n    p = argparse.ArgumentParser(description=__doc__)\n    p.add_argument(\'--prepared\', required=True)\n    p.add_argument(\'--output\', required=True)\n    p.add_argument(\'--config\', default=str(Path(__file__).parent.parent / \'configs/v1_ab.json\'))\n    p.add_argument(\'--plan\', choices=[\'A\', \'B\', \'R\'], required=True)\n    p.add_argument(\'--seed\', type=int, default=42)\n    p.add_argument(\'--max-seconds\', type=float, default=28800)\n    p.add_argument(\'--pilot-steps\', type=int, default=0)\n    p.add_argument(\'--dry-run\', action=\'store_true\')\n    args = p.parse_args()\n    if args.pilot_steps < 0:\n        p.error(\'pilot-steps must be nonnegative\')\n    run(args)\n\n\nif __name__ == \'__main__\':\n    main()\n', 'configs/v1_ab.json': '{\n  "protocol": "v1-ab-r1",\n  "batch_size": 1,\n  "grad_accum": 8,\n  "learning_rate": 0.0001,\n  "rank": 16,\n  "target_modules": "q_proj,k_proj,v_proj,o_proj",\n  "probe_every": 25,\n  "save_every": 50,\n  "train_tokens": {"knowdist": 1500000, "icldist": 500000},\n  "plans": {\n    "R": {"selection": "random", "mode": "uniform"},\n    "A": {"selection": "student", "mode": "uniform"},\n    "B": {"selection": "coverage", "mode": "uniform"}\n  },\n  "switch_policy": {\n    "selection_split": "probe",\n    "independent_report_split": "holdout",\n    "criteria": "Choose on predeclared dev metrics and total score+train cost; test must not choose A/B.",\n    "failure": "If A scoring fails, stays incomplete, loses group coverage or exceeds scoring budget, run B explicitly. Never mix checkpoints or silently substitute data.",\n    "scope": "Pilot controls, not validated hyperparameters or a novelty claim. Both plans keep fixed ICLDIST data."\n  }\n}\n'}
for name, content in SOURCES.items():
    path = PROJECT / name
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(content, encoding='utf-8')
print('Embedded source:', PROJECT)


In [ ]:
EXPERIMENT = 'qwen05-v1-ab-pilot-01'  # New tag when settings/source change
MODEL = 'Qwen/Qwen2.5-0.5B-Instruct'
MODEL_REVISION = 'main'  # tokenizer resolves and records immutable SHA
DATASET_REVISION = 'main'  # download resolves SHA before normalization
LOCAL_CORPUS = None  # Or attached raw v1 directory under /kaggle/input
LOCAL_CORPUS_REVISION = None  # Exact upstream revision, required for LOCAL_CORPUS
BENCHMARK_ROOT = None  # Attached SentiBench root; exact-overlap audit if provided
MAPPING_CONFIRMED = False  # Inspect raw records and extractor first
ACTIVE_PLAN = 'B'  # B is ready without scoring; change to A after A selection exists
RUN_SCORE_A = False
RUN_TRAIN = False
PILOT_STEPS = 50  # 0 = token-budgeted main run; pilot is NOT final comparison
SEED = 42
WORK = BASE / EXPERIMENT
NORMALIZED = WORK / 'normalized'
POOL = WORK / 'pool'
SCORES = WORK / 'scores-A'
PREPARED = WORK / 'prepared'
RUNS = WORK / ('pilot-runs' if PILOT_STEPS else 'main-runs')
WORK.mkdir(parents=True, exist_ok=True)
def run(module, *args, check=True):
    cmd = [sys.executable, str(PROJECT / 'training' / module), *map(str, args)]
    print(' '.join(cmd))
    return subprocess.run(cmd, check=check)


## Kiểm tra môi trường thực tế

Không tự thay torch/CUDA hoặc hạ phiên bản thư viện. Thiếu package thì dừng và
thiết lập môi trường Kaggle phù hợp trước khi chạy. Chỉ kiểm tra này chưa chứng
minh mọi import/model operation tương thích. Bật GPU khi tokenize/scoring/train;
normalize có thể chạy trong một notebook CPU riêng để tiết kiệm quota.


In [ ]:
packages = ['torch', 'transformers', 'peft', 'accelerate', 'huggingface-hub', 'numpy', 'safetensors']
versions = {p: importlib.metadata.version(p) for p in packages}
import torch
environment = {'python': sys.version, 'packages': versions,
    'cuda_available': torch.cuda.is_available(), 'cuda_version': torch.version.cuda,
    'gpus': [{'name': torch.cuda.get_device_name(i), 'capability': torch.cuda.get_device_capability(i),
              'total_bytes': torch.cuda.get_device_properties(i).total_memory}
             for i in range(torch.cuda.device_count())]}
(WORK / 'environment.json').write_text(json.dumps(environment, indent=2))
print(json.dumps(environment, indent=2))


## Tải v1 và kiểm tra schema

Không tải teacher/model 70B. Nếu dữ liệu đã attach, điền revision chính xác.
Xem các trường prompt/response và cách lấy văn bản cuối trong ICLDIST. Các template
ngoài các marker được hỗ trợ sẽ bị loại có lý do. Không coi phần bị loại là nhiễu
nếu chỉ chưa có extractor. Chỉ xác nhận mapping sau khi kiểm tra.


In [ ]:
if LOCAL_CORPUS:
    if not LOCAL_CORPUS_REVISION:
        raise ValueError('Provide actual LOCAL_CORPUS_REVISION')
    CORPUS, CORPUS_SHA = Path(LOCAL_CORPUS), LOCAL_CORPUS_REVISION
else:
    CORPUS = WORK / 'raw-v1'
    run('prepare_v1_pruned.py', 'download', '--revision', DATASET_REVISION, '--output', CORPUS)
    CORPUS_SHA = json.loads((CORPUS / 'download-manifest.json').read_text())['revision']
sys.path.insert(0, str(PROJECT / 'training'))
from v1_core import iter_objects
preview_files = [p for p in sorted(CORPUS.rglob('*')) if p.suffix in ('.json', '.jsonl')
                 and not p.name.endswith('manifest.json') and '.cache' not in p.parts]
for path in preview_files[:60]:
    first = next(iter_objects(path), None)
    print(path.relative_to(CORPUS), json.dumps(first, ensure_ascii=False)[:2200])


In [ ]:
if not MAPPING_CONFIRMED:
    raise ValueError('Inspect raw records, set MAPPING_CONFIRMED=True and rerun this cell')
args = ['normalize', '--input', CORPUS, '--output', NORMALIZED, '--revision', CORPUS_SHA,
        '--mapping-confirmed', '--seed', SEED]
if BENCHMARK_ROOT:
    args += ['--benchmark-root', BENCHMARK_ROOT]
run('prepare_v1_pruned.py', *args)
print((NORMALIZED / 'manifest.json').read_text())
print('Inspect exclusions.jsonl; source grouping does not isolate embedded demo text.')


In [ ]:
run('prepare_v1_pruned.py', 'tokenize', '--normalized', NORMALIZED, '--output', POOL,
    '--model', MODEL, '--model-revision', MODEL_REVISION, '--seed', SEED,
    '--know-candidates', 10000, '--icl-candidates', 5000, '--max-length', 2048)
print((POOL / 'manifest.json').read_text())
print((POOL / 'preview.json').read_text())


## Chuẩn bị B và random trước

Hai nhánh dùng chung ICLDIST và dev. Ngân sách là token, số mẫu chỉ là trần.
Nếu bị loại overlength nhiều hoặc mất nhóm, sửa candidate pool trước train.
Không tự ghi đè output; muốn đổi quy tắc phải dùng tag mới.


In [ ]:
for plan, method in [('B', 'coverage'), ('R', 'random')]:
    run('prepare_v1_pruned.py', 'select', '--pool', POOL, '--output', PREPARED / plan,
        '--method', method, '--seed', SEED)


## Chấm và chuẩn bị A nếu đủ ngân sách

Đây là proxy dùng base student, không tái lập nguyên quy trình IFD có học trước.
Khoảng IFD [0,1] là cấu hình thí nghiệm ban đầu, không chứng nhận đáp án đúng.
Score có resume, mỗi lượt tối đa 2h; lỗi/thiếu điểm không ngăn dùng B đã chuẩn bị.
Chi phí chấm phải cộng vào tổng chi phí A. Có thể rerun score với --resume.


In [ ]:
if RUN_SCORE_A:
    args = ['score', '--pool', POOL, '--output', SCORES, '--max-seconds', 7200]
    if (SCORES / 'identity.json').exists():
        args += ['--resume']
    result = run('prepare_v1_pruned.py', *args, check=False)
    summary_file = SCORES / 'summary.json'
    if result.returncode == 0 and summary_file.exists() and json.loads(summary_file.read_text())['completed']:
        result = run('prepare_v1_pruned.py', 'select', '--pool', POOL, '--scores', SCORES,
            '--method', 'student', '--output', PREPARED / 'A', '--seed', SEED, check=False)
        if result.returncode:
            print('A selection failed. Inspect coverage/scores; B remains available. Set ACTIVE_PLAN=B explicitly.')
    else:
        print('A not complete. Preserve score outputs; resume later or use B explicitly.')
else:
    print('A scoring disabled. B and R are ready.')


## Pilot hoặc train hai giai đoạn

Chạy một plan mỗi lượt; có thể đổi ACTIVE_PLAN để chạy plan khác. R/B/A có thư mục
riêng, không tái dùng adapter giữa plan. Mặc định dry-run; bật RUN_TRAIN sau kiểm tra.
Mỗi lần gọi giới hạn 8h, kiểm tra dừng tại optimizer boundary; download/probe/save
có thể tốn thêm thời gian nên theo dõi deadline phiên thực tế. Resume yêu cầu giữ
nguyên WORK, config, source và môi trường. Phục hồi full outputs vào đường dẫn
working cũ, không train từ /kaggle/input chỉ đọc. Giữ adapter KNOWDIST khi resume ICLDIST.


In [ ]:
if ACTIVE_PLAN not in ('A', 'B', 'R'):
    raise ValueError('Unknown plan')
if not (PREPARED / ACTIVE_PLAN / 'selection.json').exists():
    raise ValueError('Plan is not prepared; choose available B/R or finish A')
args = ['--prepared', PREPARED / ACTIVE_PLAN, '--output', RUNS / ACTIVE_PLAN / f'seed-{SEED}',
        '--plan', ACTIVE_PLAN, '--seed', SEED, '--max-seconds', 28800, '--pilot-steps', PILOT_STEPS]
if not RUN_TRAIN:
    args += ['--dry-run']
run('run_v1_ab.py', *args)


## Chuyển nhánh và đánh giá

Chọn A/B theo probe/dev và tổng chi phí, không theo benchmark test; holdout chỉ
báo cáo độc lập. Teacher-response CE không phải sentiment F1. Không tự kết luận A
tốt hơn nếu IFD/loss thấp hơn. Cần audited SentiBench evaluator và cùng test/demo
cho mọi checkpoint; ASQP notebook cũ đang cần audit. Không dùng old cache giữa adapters.

Để ablate fine-tuning, dùng một config mới (ví dụ mode=progress cho cả nhánh đối
chứng tương ứng), không sửa mode trong run đang resume. Trước hết giữ uniform
để tách hiệu quả chọn dữ liệu. Chưa có bảo đảm novelty hoặc F1 tăng.
